In [ ]:
workspace_id = "2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a"
bronze_lakehouse_id = "33b93b7e-0899-43c6-81a8-654bc740aa32"

In [1]:
from pyspark.sql import functions as F

bronze_taxi_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Files/nyc_taxi"

# 1. Read raw Parquet files across all ingested months from Lakehouse_Bronze
df_raw = spark.read.option("mergeSchema", "true").parquet(bronze_taxi_path)

# 2. Standardize column names to lowercase
df_standardized = df_raw.toDF(*[c.lower() for c in df_raw.columns])

# 3. Clean, validate, and enrich records (Silver layer)
df_silver = (
    df_standardized
    .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn(
        "trip_duration_minutes",
        F.round(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
            2
        )
    )
    # Retain zero-distance trips as valid business cases (waiting/cancellation charges)
    .filter(
        (F.col("tpep_pickup_datetime").isNotNull()) &
        (F.col("tpep_dropoff_datetime").isNotNull()) &
        (F.col("trip_duration_minutes") > 0) &
        (F.col("trip_duration_minutes") < 1440) &
        (F.col("trip_distance") >= 0) &
        (F.col("fare_amount") > 0) &
        (F.col("total_amount") > 0) &
        (F.col("passenger_count") > 0) &
        (F.year("tpep_pickup_datetime").between(2020, 2026))
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .dropDuplicates()
)

# 4. Write Delta table to default Lakehouse_Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_taxi_silver")
)

print(f"Successfully saved nyc_taxi_silver. Valid record count: {df_silver.count()}")
display(df_silver.limit(5))

StatementMeta(, 1f42261c-827f-4fc3-82e0-754b902e3fe1, 3, Finished, Available, Finished, False)

Zapisano tabelę nyc_taxi_silver. Liczba poprawnych rekordów: 2723735


SynapseWidget(Synapse.DataFrame, bef4e183-0965-4305-9de3-0a7bc7a170b5)